In [1]:
import numpy as np
import pandas as pd
flights = pd.read_csv('data/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_7.csv')

C:\Users\wikel\AppData\Local\Temp\ipykernel_16748\1427209745.py:3: DtypeWarning: Columns (77: Div2Airport, 84: Div2TailNum) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv('data/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_7.csv')


Sample and Trim: Cut down so remaining columns are *'FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime', 'DepDelay', 'ArrDelay', 'Cancelled', 'Diverted', 'Distance'.* 

## Why I chose these columns specifically?

*'FlightDate'*: **When** the flight happened. \
*'Reporting_Airline'*: **Which** airline. \
*'Origin, Dest'*: the route and airports. \
*'CRSDepTime'*: Scheduled departure time. \
*'Distance'*: longer flights have more room to recover from a late start. \
*'ArrDelay'*: the outcome. \
*'Cancelled, Diverted'*: tells why ArrDelay is empty and which rows to exclude. \
*'DepDelay'*: useful for analysis and cleaning checks. 



In [2]:
import random
flights['FlightDate'] = pd.to_datetime(flights['FlightDate'])
hours = [item // 100 for item in flights['CRSDepTime']]
hours = [item - 24 if item > 23 else item + 0 for item in hours ]
mins = flights['CRSDepTime'] % 100
flights['dep_hour'] = hours
flights['dep_min'] = mins
random.seed(42)
updated_flights = (flights.loc[random.sample(range((len(flights))), k=50000), ['FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'dep_hour', 'DepDelay', 'ArrDelay', 'Cancelled', 'Diverted', 'Distance']])
print(flights['FlightDate'].dtype) # ('<M8[us]') is shorthand for "datetime" data type.
updated_flights.shape

datetime64[us]


(50000, 10)

In [3]:
null_arr_delay_count = updated_flights['ArrDelay'].isna().sum() 
print(f"Amount of null arrDelays: {null_arr_delay_count}")
cancelled_flights_table = updated_flights[updated_flights['Cancelled'] == 1]
count_cancelled_flights = cancelled_flights_table['Cancelled'].sum()
print(f"Amount of flights cancelled _flights=1: {count_cancelled_flights}")
diverted_flights_table = updated_flights[updated_flights['Diverted'] == 1]
count_diverted_flights = diverted_flights_table['Diverted'].sum()
print(f"Amount of flights diverted_flights=1: {count_diverted_flights}")
updated_flights = updated_flights.dropna(subset=['ArrDelay'])


Amount of null arrDelays: 1626
Amount of flights cancelled _flights=1: 1361.0
Amount of flights diverted_flights=1: 265.0


As you can see the amount of arrDelays is the sum of the cancelled and diverted flights. Model predicts "arrivdd over 15 minutes late", and a cancelled or diverted flight has no arrival delay, so there is nothing to predict. Drop them.

In [4]:
%pip install pyarrow -U pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
binary_delays = updated_flights['ArrDelay'] > 15
print(binary_delays.mean(), flights['ArrDel15'].mean())
updated_flights['delayed'] = (binary_delays).astype(int)
updated_flights = updated_flights.drop(columns=['Cancelled', 'Diverted'])
updated_flights.to_parquet("data/flights_clean.parquet", index=False)

0.27446562202836233 0.2834154088471631


In [9]:
pd.read_parquet("data/flights_clean.parquet").shape
pd.read_parquet("data/flights_clean.parquet").dtypes

FlightDate           datetime64[us]
Reporting_Airline               str
Origin                          str
Dest                            str
dep_hour                      int64
DepDelay                    float64
ArrDelay                    float64
Distance                    float64
delayed                       int64
dtype: object